In [0]:
CATALOG = "power_prices"
G = f"{CATALOG}.gold"

# Hourly generation totals and shares per zone, reused by several aggregates
GEN_HOURLY = f"""
  SELECT zone_code, timestamp_utc, local_date,
    SUM(CASE WHEN flow_type = 'generation' THEN net_generation_mw END)                                   AS total_mw,
    SUM(CASE WHEN flow_type = 'generation' AND fuel_group = 'Solar' THEN net_generation_mw END)          AS solar_mw,
    SUM(CASE WHEN flow_type = 'generation' AND fuel_group = 'Wind' THEN net_generation_mw END)           AS wind_mw,
    SUM(CASE WHEN flow_type = 'generation' AND energy_class = 'Renewable' THEN net_generation_mw END)    AS renewable_mw,
    SUM(CASE WHEN flow_type = 'generation' AND energy_class = 'Fossil' THEN net_generation_mw END)       AS fossil_mw,
    SUM(CASE WHEN flow_type = 'generation' AND energy_class = 'Nuclear' THEN net_generation_mw END)      AS nuclear_mw
  FROM {G}.fact_generation_hourly
  GROUP BY zone_code, timestamp_utc, local_date
"""

# Zone-days that have every local hour (24, or 23/25 on clock-change days)
COMPLETE_PRICE_DAYS = f"""
  SELECT p.zone_code, p.local_date
  FROM {G}.fact_price_hourly p
  JOIN {G}.dim_date d ON p.date_key = d.date_key
  GROUP BY p.zone_code, p.local_date
  HAVING COUNT(*) >= MAX(d.hours_in_day)
"""

aggregates = {}

# ---------------------------------------------------------------
# 1. agg_price_daily: zone x day, complete days only (pages 1, 2, 5)
# ---------------------------------------------------------------
aggregates["agg_price_daily"] = f"""
CREATE OR REPLACE TABLE {G}.agg_price_daily
COMMENT 'Daily price statistics per zone (complete days only) with load and generation shares; feeds Market Overview, Negative Prices, Generation Mix'
AS
WITH p AS (
  SELECT p.zone_code, p.local_date, p.local_hour, p.price_eur_mwh, p.is_negative, d.is_weekend, d.hours_in_day
  FROM {G}.fact_price_hourly p
  JOIN {G}.dim_date d ON p.date_key = d.date_key
),
price_day AS (
  SELECT zone_code, local_date AS date,
    ROUND(AVG(price_eur_mwh), 2)                                                                   AS avg_price_eur_mwh,
    ROUND(MIN(price_eur_mwh), 2)                                                                   AS min_price_eur_mwh,
    ROUND(MAX(price_eur_mwh), 2)                                                                   AS max_price_eur_mwh,
    ROUND(MAX(price_eur_mwh) - MIN(price_eur_mwh), 2)                                              AS intraday_spread_eur_mwh,
    ROUND(AVG(CASE WHEN NOT is_weekend AND local_hour BETWEEN 8 AND 19 THEN price_eur_mwh END), 2) AS peak_price_eur_mwh,
    ROUND(AVG(CASE WHEN is_weekend OR local_hour NOT BETWEEN 8 AND 19 THEN price_eur_mwh END), 2)  AS offpeak_price_eur_mwh,
    ROUND(AVG(CASE WHEN local_hour BETWEEN 10 AND 15 THEN price_eur_mwh END), 2)                   AS solar_window_price_eur_mwh,
    ROUND(AVG(CASE WHEN local_hour BETWEEN 17 AND 20 THEN price_eur_mwh END), 2)                   AS evening_peak_price_eur_mwh,
    CAST(SUM(CASE WHEN is_negative THEN 1 ELSE 0 END) AS INT)                                      AS negative_hours,
    CAST(COUNT(*) AS INT)                                                                          AS price_hours
  FROM p
  GROUP BY zone_code, local_date
  HAVING COUNT(*) >= MAX(hours_in_day)
),
gen_day AS (
  SELECT zone_code, local_date AS date,
    SUM(total_mw) AS gen_mwh, SUM(solar_mw) AS solar_mwh, SUM(wind_mw) AS wind_mwh,
    SUM(renewable_mw) AS renewable_mwh, SUM(fossil_mw) AS fossil_mwh, SUM(nuclear_mw) AS nuclear_mwh
  FROM ({GEN_HOURLY})
  GROUP BY zone_code, local_date
),
load_day AS (
  SELECT zone_code, local_date AS date, SUM(load_actual_mw) AS load_mwh
  FROM {G}.fact_load_hourly
  GROUP BY zone_code, local_date
)
SELECT pd.*,
  ROUND(l.load_mwh, 0)                                AS load_mwh,
  ROUND(g.gen_mwh, 0)                                 AS generation_mwh,
  ROUND(try_divide(g.renewable_mwh, g.gen_mwh), 4)    AS renewable_share,
  ROUND(try_divide(g.solar_mwh, g.gen_mwh), 4)        AS solar_share,
  ROUND(try_divide(g.wind_mwh, g.gen_mwh), 4)         AS wind_share,
  ROUND(try_divide(g.fossil_mwh, g.gen_mwh), 4)       AS fossil_share,
  ROUND(try_divide(g.nuclear_mwh, g.gen_mwh), 4)      AS nuclear_share
FROM price_day pd
LEFT JOIN gen_day g  USING (zone_code, date)
LEFT JOIN load_day l USING (zone_code, date)
"""

# ---------------------------------------------------------------
# 2. agg_price_profile: zone x year x month x local hour (pages 2, 3)
# ---------------------------------------------------------------
aggregates["agg_price_profile"] = f"""
CREATE OR REPLACE TABLE {G}.agg_price_profile
COMMENT 'Average price, negative hours and solar share by zone, year, month and local hour; feeds Negative Prices and The Solar Dip'
AS
WITH gen AS ({GEN_HOURLY}),
h AS (
  SELECT p.zone_code, year(p.local_date) AS year, month(p.local_date) AS month, p.local_hour,
         p.price_eur_mwh, p.is_negative, g.solar_mw, g.total_mw
  FROM {G}.fact_price_hourly p
  LEFT JOIN gen g ON p.zone_code = g.zone_code AND p.timestamp_utc = g.timestamp_utc
)
SELECT zone_code, year, month, local_hour,
  ROUND(AVG(price_eur_mwh), 2)                               AS avg_price_eur_mwh,
  CAST(SUM(CASE WHEN is_negative THEN 1 ELSE 0 END) AS INT)  AS negative_hours,
  CAST(COUNT(*) AS INT)                                      AS price_hours,
  ROUND(try_divide(SUM(solar_mw), SUM(total_mw)), 4)         AS solar_share
FROM h
GROUP BY zone_code, year, month, local_hour
"""

# ---------------------------------------------------------------
# 3. agg_generation_monthly: zone x month x fuel group (page 5)
# ---------------------------------------------------------------
aggregates["agg_generation_monthly"] = f"""
CREATE OR REPLACE TABLE {G}.agg_generation_monthly
COMMENT 'Monthly generation per zone and fuel group with share of total generation; feeds Generation Mix'
AS
WITH m AS (
  SELECT zone_code, trunc(local_date, 'MM') AS month_start, fuel_group,
         SUM(net_generation_mw)                                                 AS net_mwh,
         SUM(CASE WHEN flow_type = 'generation' THEN net_generation_mw END)     AS gen_mwh
  FROM {G}.fact_generation_hourly
  GROUP BY zone_code, trunc(local_date, 'MM'), fuel_group
),
t AS (
  SELECT zone_code, month_start, SUM(gen_mwh) AS total_mwh
  FROM m GROUP BY zone_code, month_start
)
SELECT m.zone_code, m.month_start, m.fuel_group,
  ROUND(m.net_mwh, 0)                             AS generation_mwh,
  ROUND(try_divide(m.gen_mwh, t.total_mwh), 4)    AS share_of_generation
FROM m
JOIN t ON m.zone_code = t.zone_code AND m.month_start = t.month_start
"""

# ---------------------------------------------------------------
# 4. agg_border_monthly: border x month (page 4)
# ---------------------------------------------------------------
aggregates["agg_border_monthly"] = f"""
CREATE OR REPLACE TABLE {G}.agg_border_monthly
COMMENT 'Monthly flows, price spread and share of converged hours per border; feeds Flows and Convergence'
AS
WITH j AS (
  SELECT f.border_key, f.zone_a, f.zone_b, f.timestamp_utc,
         f.flow_a_to_b_mw, f.flow_b_to_a_mw, f.net_flow_a_to_b_mw,
         CAST(date_trunc('MONTH', from_utc_timestamp(f.timestamp_utc, 'Europe/Brussels')) AS DATE) AS month_start,
         pa.price_eur_mwh AS price_a, pb.price_eur_mwh AS price_b
  FROM {G}.fact_flow_hourly f
  LEFT JOIN {G}.fact_price_hourly pa ON pa.zone_code = f.zone_a AND pa.timestamp_utc = f.timestamp_utc
  LEFT JOIN {G}.fact_price_hourly pb ON pb.zone_code = f.zone_b AND pb.timestamp_utc = f.timestamp_utc
)
SELECT border_key, zone_a, zone_b, month_start,
  ROUND(AVG(flow_a_to_b_mw), 1)                                     AS avg_flow_a_to_b_mw,
  ROUND(AVG(flow_b_to_a_mw), 1)                                     AS avg_flow_b_to_a_mw,
  ROUND(AVG(net_flow_a_to_b_mw), 1)                                 AS avg_net_flow_a_to_b_mw,
  ROUND(SUM(net_flow_a_to_b_mw) / 1000, 1)                          AS net_flow_a_to_b_gwh,
  ROUND(AVG(price_a), 2)                                            AS avg_price_a_eur_mwh,
  ROUND(AVG(price_b), 2)                                            AS avg_price_b_eur_mwh,
  ROUND(AVG(ABS(price_a - price_b)), 2)                             AS avg_abs_spread_eur_mwh,
  ROUND(try_divide(SUM(CASE WHEN ABS(price_a - price_b) < 0.01 THEN 1 ELSE 0 END),
                   COUNT(price_a - price_b)), 4)                    AS converged_share,
  CAST(COUNT(*) AS INT)                                             AS flow_hours
FROM j
GROUP BY border_key, zone_a, zone_b, month_start
"""

# ---------------------------------------------------------------
# 5. agg_tomorrow_hourly: latest complete delivery day and the day before (page 6)
# ---------------------------------------------------------------
aggregates["agg_tomorrow_hourly"] = f"""
CREATE OR REPLACE TABLE {G}.agg_tomorrow_hourly
COMMENT 'Hourly day-ahead prices for the latest complete delivery day and the day before, with cheapest and priciest hour ranks; feeds Tomorrow''s Prices'
AS
WITH latest AS (SELECT MAX(local_date) AS d FROM ({COMPLETE_PRICE_DAYS})),
h AS (
  SELECT p.zone_code, p.local_date AS delivery_date, p.local_hour, AVG(p.price_eur_mwh) AS price
  FROM {G}.fact_price_hourly p
  JOIN latest ON p.local_date BETWEEN date_sub(latest.d, 1) AND latest.d
  GROUP BY p.zone_code, p.local_date, p.local_hour
)
SELECT h.zone_code, h.delivery_date,
  h.delivery_date = latest.d                                                                AS is_latest_day,
  h.local_hour,
  ROUND(h.price, 2)                                                                         AS price_eur_mwh,
  CAST(RANK() OVER (PARTITION BY h.zone_code, h.delivery_date ORDER BY h.price) AS INT)     AS cheapest_rank,
  CAST(RANK() OVER (PARTITION BY h.zone_code, h.delivery_date ORDER BY h.price DESC) AS INT) AS priciest_rank
FROM h CROSS JOIN latest
"""

# ---------------------------------------------------------------
# 6. agg_refresh_info: one row of data freshness (every page)
# ---------------------------------------------------------------
aggregates["agg_refresh_info"] = f"""
CREATE OR REPLACE TABLE {G}.agg_refresh_info
COMMENT 'One row: when gold was rebuilt and the latest data available per dataset; feeds the Data as of line'
AS
SELECT
  current_timestamp()                                              AS refreshed_at_utc,
  (SELECT MAX(local_date) FROM ({COMPLETE_PRICE_DAYS}))            AS latest_price_date,
  (SELECT MAX(timestamp_utc) FROM {G}.fact_load_hourly)            AS latest_load_utc,
  (SELECT MAX(timestamp_utc) FROM {G}.fact_generation_hourly)      AS latest_generation_utc,
  (SELECT MAX(timestamp_utc) FROM {G}.fact_flow_hourly)            AS latest_flow_utc,
  (SELECT COUNT(DISTINCT zone_code) FROM {G}.fact_price_hourly)    AS zones_with_prices
"""

# ---------------------------------------------------------------
# Build and check
# ---------------------------------------------------------------
spark.conf.set("spark.sql.session.timeZone", "UTC")
total_cells = 0
print(f"{'table':<24} {'rows':>9} {'cols':>5} {'cells':>11}")
for name, sql in aggregates.items():
    spark.sql(sql)
    t = spark.table(f"{G}.{name}")
    rows, cols = t.count(), len(t.columns)
    total_cells += rows * cols
    print(f"{name:<24} {rows:>9,} {cols:>5} {rows * cols:>11,}")
print(f"{'TOTAL':<24} {'':>9} {'':>5} {total_cells:>11,}   (Google Sheets limit: 10,000,000)")